### Step 3a: Add discounts

Same model as Step 2, with one new idea: **the station can offer a discount.**

In Step 2, the only way to lower a customer's CPM was to guarantee more audience, which adds makegood risk. Only 8 of 30 prospects could buy.
A discount is a second way to lower the CPM. It costs revenue up front, but adds no risk.

| Way to lower the CPM | What it costs |
| --- | --- |
| Guarantee more audience | Makegood risk (depends on the customer's 2x/3x/4x multiplier) |
| Discount the price | Revenue, for certain |

**What changed:**
- New set: `discounts = [0.00, 0.05, 0.10]` (team assumption, same as Plan B).
- `price` and `cpm` now depend on the package **and** the discount: price = \$108,000 × (1 − discount), CPM = price / audience promised.
- The decision variable gets one more index: `x[c,p,d]` = 1 if customer c gets package p at discount d.
- Everything else follows the same pattern as Step 2, with `d` added to each loop.

**Size:** 30 customers × 4 packages × 3 discounts = **360 decision variables** and **391 constraints**.

**CPM for every option** (price / audience promised):

| Package | Guarantee | 0% off | 5% off | 10% off |
| --- | ---: | ---: | ---: | ---: |
| conservative | +0% | \$66.20 | \$62.89 | \$59.58 |
| medium | +15% | \$57.57 | \$54.69 | \$51.81 |
| premium | +20% | \$55.17 | \$52.41 | \$49.65 |
| aggressive | +25% | \$52.96 | \$50.32 | \$47.67 |

**Assumptions (to confirm with Ed):**
1. Discount options of 0%, 5%, and 10%.
2. The makegood is valued at the CPM the customer actually paid (after the discount).

In [ ]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import course_utils  # course helpers (print_status, print_model, ...)

In [ ]:
# ============================================================
# LOAD THE DATA
# ============================================================

# Ed's spreadsheet. Keep it in the same folder as this notebook.
# We only read from it; we never edit it.

DATA_FILE = "NFL Ad PKG MAIN.xlsx"


# ------------------------------------------------------------
# A. The package: 'Pre-Schedule Release', rows 13-17
# ------------------------------------------------------------

# One row per game type (Thanksgiving, noon regional, ...).
# This is what the station knows in July, before the schedule is out.
#
# col C = game type
# col E = number of games
# col F = rate per spot ($)
# col H = audience per game (000s = thousands of households)

df_games = pd.read_excel(
    DATA_FILE,
    sheet_name="Pre-Schedule Release",
    header=None,
    skiprows=12,
    nrows=5,
    usecols="C,E,F,H"
)

df_games.columns = ["game_type", "num_games", "rate", "audience_000s"]

package_price = (df_games["num_games"] * df_games["rate"]).sum()
baseline_000s = (df_games["num_games"] * df_games["audience_000s"]).sum()

# Check against the spreadsheet's own totals (row 18)
assert df_games["num_games"].sum() == 27
assert package_price == 108_000
assert abs(baseline_000s - 1631.3) < 0.001


# ------------------------------------------------------------
# B. Ratings scenarios: 'Schedule Release 1-Min Qualifer', T6:U8
# ------------------------------------------------------------

# Nielsen's new 1-minute rule raises ratings, but nobody knows
# by how much. Management gives 3 possible increases.
#
# col T = label, such as "15% Increase"
# col U = probability of that increase

df_scen = pd.read_excel(
    DATA_FILE,
    sheet_name="Schedule Release 1-Min Qualifer",
    header=None,
    skiprows=5,
    nrows=3,
    usecols="T:U"
)

df_scen.columns = ["label", "probability"]

# "15% Increase" -> 0.15
df_scen["increase"] = df_scen["label"].str.split("%").str[0].astype(float) / 100

# Check: probabilities add up to 100%
assert abs(df_scen["probability"].sum() - 1.0) < 0.000001


# ------------------------------------------------------------
# C. Prospects: 'Prospect Data Table', rows 4-33
# ------------------------------------------------------------

# col B = prospect ID
# col C = prospect name
# col D = salesperson        (not used until a later step)
# col E = close ratio        (not used until a later step)
# col F = goal CPM
# col G = makegood penalty multiplier (2, 3, or 4)

df_p = pd.read_excel(
    DATA_FILE,
    sheet_name="Prospect Data Table",
    skiprows=2,
    nrows=30,
    usecols="B:G"
)

df_p.columns = ["id", "prospect", "salesperson", "close_ratio",
                "goal_cpm", "penalty"]

# Check: 30 prospects, 6 salespeople with 5 prospects each
assert len(df_p) == 30
assert df_p["salesperson"].value_counts().tolist() == [5] * 6


print(f"Package price:  ${package_price:,.0f}")
print(f"Baseline 000s:  {baseline_000s:,.1f}")
print()
print(df_scen)
print()
df_p

In [ ]:
# ============================================================
# 1. INDEX SETS
# ============================================================

# customers: the 30 prospects, read from the spreadsheet

customers = df_p["id"].tolist()

# packages: how much audience we guarantee, above the baseline.
# Same 4 names as the base model. The levels match the
# 3 possible Nielsen increases, plus no increase at all.

packages = [
    "conservative",
    "medium",
    "premium",
    "aggressive"
]

# discounts: how much we take off the $108,000 price (NEW)

discounts = [0.00, 0.05, 0.10]

# scenarios: the 3 possible Nielsen increases, from the spreadsheet

scenarios = df_scen["label"].tolist()


# ============================================================
# 2. CUSTOMER PARAMETERS
# ============================================================

goal_cpm = dict(zip(df_p["id"], df_p["goal_cpm"]))

# Makegood multiplier: if we fall short, customer c is owed
# penalty[c] times the value of the missing audience.

underdelivery_penalty = dict(zip(df_p["id"], df_p["penalty"]))


# ============================================================
# 3. AD PACKAGE PARAMETERS
# ============================================================

# Guarantee level for each package (TEAM ASSUMPTION)

guarantee = {
    "conservative": 0.00,
    "medium": 0.15,
    "premium": 0.20,
    "aggressive": 0.25
}

# Audience we promise the customer (000s)

guaranteed_000s = {}

for p in packages:
    guaranteed_000s[p] = baseline_000s * (1 + guarantee[p])

# Price and CPM now depend on the package AND the discount.
# A bigger guarantee or a bigger discount means a lower CPM.

price = {}
cpm = {}

for p in packages:
    for d in discounts:
        price[p, d] = package_price * (1 - d)
        cpm[p, d] = price[p, d] / guaranteed_000s[p]


# ============================================================
# 4. ACTUAL AUDIENCE OUTCOMES
# ============================================================

# 3 separate outcomes that already add up to 100%.
#
# Actual audience = baseline x (1 + Nielsen increase)

probability = dict(zip(df_scen["label"], df_scen["probability"]))

actual_000s = {}

for s in scenarios:
    increase = df_scen.loc[df_scen["label"] == s, "increase"].iloc[0]
    actual_000s[s] = baseline_000s * (1 + increase)


# ============================================================
# 5. DETERMINE PACKAGE ELIGIBILITY
# ============================================================

# eligible[c,p,d] = 1 if customer c's goal CPM is high enough
# to accept package p at discount d.
#
# Otherwise eligible[c,p,d] = 0.

eligible = {}

for c in customers:
    for p in packages:
        for d in discounts:

            if cpm[p, d] <= goal_cpm[c]:
                eligible[c, p, d] = 1
            else:
                eligible[c, p, d] = 0


# ============================================================
# 6. CALCULATE EXPECTED UNDERDELIVERY PENALTY
# ============================================================

# Shortfall (000s) x CPM the customer paid ($ per 000) = value
# of the missing audience in dollars. The customer is owed that
# value times their penalty multiplier.

expected_penalty = {}

for c in customers:
    for p in packages:
        for d in discounts:

            expected_penalty[c, p, d] = sum(
                probability[s] *
                max(guaranteed_000s[p] - actual_000s[s], 0) *
                cpm[p, d] *
                underdelivery_penalty[c]
                for s in scenarios
            )


# ============================================================
# 7. CALCULATE EXPECTED PROFIT
# ============================================================

expected_profit = {}

for c in customers:
    for p in packages:
        for d in discounts:

            expected_profit[c, p, d] = (
                price[p, d]
                - expected_penalty[c, p, d]
            )


# ============================================================
# 8. CREATE GUROBI MODEL
# ============================================================

model = gp.Model("NFL_Advertising_Optimization_Discounts")


# ============================================================
# 9. DECISION VARIABLES
# ============================================================

# x[c,p,d] = 1 if customer c receives package p at discount d
#            0 otherwise

x = model.addVars(
    customers,
    packages,
    discounts,
    vtype=GRB.BINARY,
    name="x"
)


# ============================================================
# 10. CONSTRAINTS
# ============================================================

# ------------------------------------------------------------
# Constraint 1:
# Each customer can receive at most one package
# (one guarantee level at one discount).
# ------------------------------------------------------------

model.addConstrs(
    (
        gp.quicksum(x[c, p, d] for p in packages for d in discounts) <= 1
        for c in customers
    ),
    name="one_package_per_customer"
)


# ------------------------------------------------------------
# Constraint 2:
# Package CPM cannot exceed customer's goal CPM.
# ------------------------------------------------------------

model.addConstrs(
    (
        x[c, p, d] <= eligible[c, p, d]
        for c in customers
        for p in packages
        for d in discounts
    ),
    name="cpm_requirement"
)


# ------------------------------------------------------------
# Constraint 3:
# We have 20 advertising spots, so we can sell at most 20 packages.
# ------------------------------------------------------------

model.addConstr(
    gp.quicksum(
        x[c, p, d]
        for c in customers
        for p in packages
        for d in discounts
    ) <= 20,
    name="advertising_spots"
)


# ============================================================
# 11. OBJECTIVE FUNCTION
# ============================================================

# Maximize expected profit:
#
# Revenue from the package (after discount)
# minus
# Expected underdelivery penalty

model.setObjective(
    gp.quicksum(
        expected_profit[c, p, d] * x[c, p, d]
        for c in customers
        for p in packages
        for d in discounts
    ),
    GRB.MAXIMIZE
)


# ============================================================
# 12. SOLVE MODEL
# ============================================================

model.optimize()


# ============================================================
# 13. DISPLAY RESULTS
# ============================================================

print("\nPACKAGE OPTIONS (CPM by guarantee and discount)")
print("=" * 75)

for p in packages:

    line = f"{p.capitalize():12} | Guarantee: +{guarantee[p]:>4.0%} |"

    for d in discounts:

        num_eligible = sum(eligible[c, p, d] for c in customers)

        line += f" {d:>4.0%} off: ${cpm[p, d]:>5.2f} ({num_eligible:>2} eligible) |"

    print(line)

if model.status == GRB.OPTIMAL:

    print("\nOPTIMAL ADVERTISING PLAN")
    print("=" * 75)

    total_revenue = 0
    total_expected_penalty = 0
    packages_sold = 0

    for c in customers:
        for p in packages:
            for d in discounts:

                if x[c, p, d].X > 0.5:

                    revenue = price[p, d]
                    penalty = expected_penalty[c, p, d]
                    profit = expected_profit[c, p, d]

                    total_revenue += revenue
                    total_expected_penalty += penalty
                    packages_sold += 1

                    print(
                        f"{c}: "
                        f"{p.capitalize():12} | "
                        f"Discount: {d:>4.0%} | "
                        f"Goal CPM: ${goal_cpm[c]:>5.2f} | "
                        f"CPM: ${cpm[p, d]:>5.2f} | "
                        f"Penalty: {underdelivery_penalty[c]}x | "
                        f"Revenue: ${revenue:>9,.0f} | "
                        f"Expected Penalty: ${penalty:>9,.2f} | "
                        f"Expected Profit: ${profit:>10,.2f}"
                    )

    print("=" * 75)

    print(f"Packages Sold:       {packages_sold} of 20")
    print(f"Total Revenue:       ${total_revenue:,.2f}")
    print(f"Expected Penalties:  ${total_expected_penalty:,.2f}")
    print(f"Expected Profit:     ${model.ObjVal:,.2f}")

else:

    print("No optimal solution found.")

### What the results show

- **17 packages sell, up from 8.** Expected profit rises from \$776,934 to \$1,609,889.
- **Every prospect who can qualify buys.** 17 prospects have a goal CPM at or above the cheapest option (\$47.67). The other 13 can't be reached with these discounts and guarantees, so 3 of the 20 spots go unused.
- **The model trades discount against guarantee, customer by customer.** The choice depends on the customer's penalty multiplier:
  - **C26 (2x penalty)** takes premium at no discount: \$108,000 − \$4,500 expected penalty = **\$103,500**. A 5% discount on medium would give only \$102,600. With a low multiplier, the risk is cheaper than the discount.
  - **C25 (4x penalty)** takes medium at 5% off: **\$102,600** with no risk. Premium at no discount would give \$108,000 − \$9,000 = \$99,000. With a high multiplier, the discount is cheaper than the risk.
- **Medium is the workhorse.** It's risk-free (the smallest Nielsen increase is 15%), so 10 customers get medium at whatever discount they need.
- **Aggressive is the last resort.** The 4 customers on aggressive (C11-C14) have the lowest goals that can still be reached. Only the biggest guarantee plus the biggest discount gets there.
- **C11 barely qualifies** (goal \$47.67 vs. CPM \$47.668), like C29 in Step 2.

**Worked example (C16, premium at 10% off):** the expected shortfall is 50% × 81.57 = 40.78 (000s) (only the +15% scenario falls short). At the \$49.65 CPM it paid, that's worth \$2,025, and C16's 3x multiplier makes it **\$6,075**.

### Why discounts matter

**Discounts are what turn this into a real decision.** Without them (Step 2), the model has only one way to lower a customer's CPM, a bigger guarantee, so the answer is a single rule: give each customer the smallest guarantee that meets their goal. Only 8 prospects qualify, and the only way to reduce risk is to drop customers. There is no trade-off.

With discounts, there are two ways to lower the CPM:
- **Promise more audience:** costs makegood risk, which depends on the customer's multiplier.
- **Charge less:** costs revenue, for certain.

The model picks the cheaper one for each customer. That's why C26 (2x) takes the risk and C25 (4x) takes the discount. It's also what gives the revenue-vs-risk trade-off in a later step its meaning: to cut risk, the station moves customers from "big guarantee, full price" to "safe guarantee, a bit off."

### Changes from Step 2

1. New set `discounts`, and a new index `d` on `price`, `cpm`, `eligible`, `expected_penalty`, `expected_profit`, and `x`.
2. Constraint 1 now sums over packages **and** discounts: each customer still gets at most one option.
3. The display shows the CPM and number of eligible customers for every option, and the discount each customer gets.
4. The data cell is unchanged.